```bash
CUDA_VISIBLE_DEVICES=7 vllm serve Qwen/Qwen2.5-7B-Instruct \
    --host 0.0.0.0 \
    --port 8084 \
    --gpu-memory-utilization 0.85 \
    --enable-prefix-caching \
    --dtype bfloat16 \
    --max_model_len 32000 \
    --trust-remote-code
```

```bash
CUDA_VISIBLE_DEVICES=6 vllm serve Skywork/Skywork-o1-Open-PRM-Qwen-2.5-7B \
    --host 0.0.0.0 \
    --port 8082 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

```bash
CUDA_VISIBLE_DEVICES=5 vllm serve Qwen/Qwen2.5-Math-PRM-7B \
    --host 0.0.0.0 \
    --port 8083 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

In [95]:
from openai import OpenAI

OPENAI_API_KEY = "EMPTY"
OPENAI_API_BASE = "http://localhost:{PORT}/v1"

causal_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE.format(PORT=8084),
)
causal_model = causal_client.models.list().data[0].id

skywork_prm_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE.format(PORT=8082),
)
skywork_prm_model = skywork_prm_client.models.list().data[0].id

qwen_prm_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE.format(PORT=8083),
)
qwen_prm_model = qwen_prm_client.models.list().data[0].id


In [96]:
import pandas as pd
from constants.prompts_constants import (
    VERBOSE_TASK, CONSISE_TASK, EQ_TO_TEXT_TASK, CHANGE_NUMBERS_TASK
)
from utils.attack_utils import augmentor, equivalence_check, prm_scorer


In [100]:
def attack(df_sample, 
    task_text, 
    experiment_name,
    augment_question=True,
    prm_clients = [skywork_prm_client, qwen_prm_client],
    causal_client=causal_client
):
    causal_model = causal_client.models.list().data[0].id
    prm_models = [prm_client.models.list().data[0].id for prm_client in prm_clients]

    # Apply the augmentor function to each row in the DataFrame
    aug_results = []
    aug_results = augmentor(df_sample, task_text, client=causal_client, model=causal_model)
    df_aug = pd.DataFrame(aug_results)

    # Check equivalence
    equivalence_results = []
    equivalence_results = equivalence_check(df_sample, df_aug, client=causal_client, model=causal_model)
    df_aug["equivalence"] = equivalence_results["equivalence"]
    df_aug["body_equivalence_results"] = equivalence_results["body_equivalence_results"]

    # PRM Scorer
    for prm_client, prm_model in zip(prm_clients, prm_models):
        rewards = prm_scorer(questions=df_aug["aug_problem"].tolist() if augment_question else df_sample["problem"].tolist(),
                            steps=df_aug["aug_steps"].tolist(), 
                            client=prm_client, model=prm_model)
        df_aug[f"{prm_model}--aug_rewards"] = rewards
    
    # concat the original and augmented DataFrames
    for key in df_aug.keys():
        df_sample[key] = df_aug[key]
    
    # Save the DataFrame to a CSV file
    df_sample.to_parquet(f"experiments/{experiment_name}.parquet", index=False)
    return df_sample

In [97]:
df = pd.read_parquet("data/processbench.parquet")


## TODO: Filter dataset
sample_size = 2
df_sample = df.sample(sample_size, random_state=42).reset_index(drop=True)


Index(['id', 'generator', 'problem', 'steps', 'final_answer_correct', 'label',
       'split', 'steps_len', 'per_step_len', 'Qwen2.5-Math-PRM-7B',
       'Skywork-o1-Open-PRM-Qwen-2.5-7B'],
      dtype='object')

In [ ]:
df_sample = attack(df, 
    task_text=VERBOSE_TASK, 
    prm_client=skywork_prm_client, 
    experiment_name="skywork_verbose_task_2"
)

  0%|          | 0/14 [00:00<?, ?it/s]

In [17]:
index = 4

problem, steps = df_sample.iloc[index]["problem"], df_sample.iloc[index]["steps"]
augmented_problem, augmented_steps = df_sample.iloc[index]["aug_problem"], df_sample.iloc[index]["aug_steps"]
equivalence = df_sample.iloc[index]["equivalence"]
print("\nOriginal Question:")
print("-" * 80)
print(problem)

print("\nOriginal Steps:")
print("-" * 80)
print(steps)

print("\nOriginal Rewards:")
print("-" * 80)
print(df_sample.iloc[index]["Skywork-o1-Open-PRM-Qwen-2.5-7B"])
print(df_sample.iloc[index]["Qwen2.5-Math-PRM-7B"])

print("\nAugmented Question:")
print("-" * 80)
print(augmented_problem)

print("\nAugmented Steps:")
print("-" * 80)
print(augmented_steps)

print("\nAugmented Rewards:")
print("-" * 80)
print(df_sample.iloc[index]["Skywork/Skywork-o1-Open-PRM-Qwen-2.5-7B--aug_rewards"])
print(df_sample.iloc[index]["Qwen/Qwen2.5-Math-PRM-7B--aug_rewards"])

print("\nEquivalence Check:")
print("-" * 80)
print(f"Original and augmented versions are equivalent: {equivalence}")




Original Question:
--------------------------------------------------------------------------------
Four semi-circles are shown with $AB:BC:CD = 1:2:3$. What is the ratio of the shaded area to the unshaded area in the semi circle with diameter $AD$? Express your answer as a common fraction. [asy]
import olympiad; import geometry; size(150); defaultpen(linewidth(0.8));
filldraw(arc((6,0),6,0,180)--cycle);
filldraw(arc((3,0),3,0,180)--cycle,fillpen=white); filldraw(arc((8,0),2,0,180)--cycle,fillpen=white); filldraw(arc((11,0),1,0,180)--cycle,fillpen=white);
label("$A$",(12,0),S); label("$B$",(10,0),S); label("$C$",(6,0),S); label("$D$",(0,0),S);
[/asy]

Original Steps:
--------------------------------------------------------------------------------
['To find the ratio of the shaded area to the unshaded area in the semi-circle with diameter AD, we need to first calculate the areas of each semi-circle.'
 'First, determine the radius of each semi-circle. The radius of the semi-circle with 